:::{warning}
Currently the main user-interface is the `StateTransitionManager`. There is work in progress to remove it and split its functionality into several functions/classes to separate concerns
and to facilitate the modification of intermediate results like the filtering of `QNProblemSet`s, setting allowed interaction types, etc. (see below)
:::


# Visualize solutions


The {mod}`~qrules.io` module allows you to convert {class}`.MutableTransition`, {class}`.Topology` instances, and {class}`.ProblemSet`s to [Mermaid Markdown language](https://mermaid.ai/open-source/syntax/flowchart.html) with {func}`.asmermaid`. You can visualize the instances with {func}`.show_mermaid_markdown` which runs {func}`.asmermaid` itself when being called. This is particularly useful after running {meth}`~.StateTransitionManager.find_solutions`, which produces a {class}`.ReactionInfo` object with a {class}`.list` of {class}`.MutableTransition` instances (see {doc}`/usage/reaction`).


## Topologies


First of all, here are is an example of how to visualize a group of {class}`.Topology` instances. We use {func}`.create_isobar_topologies` and {func}`.create_n_body_topology` to create a few standard topologies.


In [1]:
import qrules
from qrules.io import show_mermaid_markdown
from qrules.conservation_rules import (
    parity_conservation,
    spin_magnitude_conservation,
    spin_validity,
)
from qrules.particle import Spin
from qrules.quantum_numbers import EdgeQuantumNumbers, NodeQuantumNumbers
from qrules.solving import (
    CSPSolver,
    dict_set_intersection,
    filter_quantum_number_problem_set,
)
from qrules.topology import create_isobar_topologies, create_n_body_topology
from qrules.transition import State

In [2]:
topology = create_n_body_topology(2, 4)
show_mermaid_markdown(topology, render_initial_state_id=True)

```mermaid
flowchart LR
    n_0["0"]
    n_1["1"]
    n_2["2"]
    n_3["3"]
    A["-1"]
    B["-2"]
    N0
    B --> N0
    A --> N0
    N0 --> n_0
    N0 --> n_1
    N0 --> n_2
    N0 --> n_3

```

Note the IDs of the {attr}`~.Topology.nodes` is also rendered if there is more than node:


In [3]:
topologies = create_isobar_topologies(4)
show_mermaid_markdown(topologies)

```mermaid
flowchart LR
    T0_0["0"]
    T0_1["1"]
    T0_2["2"]
    T0_3["3"]
    T0_A
    T0_N0["(0)"]
    T0_N1["(1)"]
    T0_N2["(2)"]
    T0_A --> T0_N0
    T0_N0 --> T0_N1
    T0_N0 --> T0_N2
    T0_N1 --> T0_0
    T0_N1 --> T0_1
    T0_N2 --> T0_2
    T0_N2 --> T0_3
    T1_0["0"]
    T1_1["1"]
    T1_2["2"]
    T1_3["3"]
    T1_A
    T1_N0["(0)"]
    T1_N1["(1)"]
    T1_N2["(2)"]
    T1_A --> T1_N0
    T1_N0 --> T1_N1
    T1_N0 --> T1_0
    T1_N1 --> T1_N2
    T1_N1 --> T1_1
    T1_N2 --> T1_2
    T1_N2 --> T1_3

```

This can be turned on or off with the arguments of {func}`.show_mermaid_markdown`:


In [4]:
topologies = create_isobar_topologies(3)
show_mermaid_markdown(topologies, render_node=False)

```mermaid
flowchart LR
    T0_0["0"]
    T0_1["1"]
    T0_2["2"]
    T0_A
    T0_N0
    T0_N1
    T0_A --> T0_N0
    T0_N0 --> T0_N1
    T0_N0 --> T0_0
    T0_N1 --> T0_1
    T0_N1 --> T0_2

```

{func}`.show_mermaid_markdown` provides other options as well:


In [5]:
topologies = create_isobar_topologies(5)
show_mermaid_markdown(topologies[0], 
                      render_final_state_id=False, 
                      render_resonance_id=True, 
                      render_node=False)

```mermaid
flowchart LR
    n_0
    n_1
    n_2
    n_3
    n_4
    A
    N0
    N1
    N2
    N3
    A --> N0
    N0 -->|5| N1
    N0 --> n_0
    N1 -->|6| N2
    N1 --> n_1
    N2 -->|7| N3
    N2 --> n_2
    N3 --> n_3
    N3 --> n_4

```

(problem-sets)=

## {class}`.ProblemSet`s


As noted in {doc}`reaction`, the {class}`.StateTransitionManager` provides more control than the façade function {func}`.generate_transitions`. One advantages, is that the {class}`.StateTransitionManager` first generates a set of {class}`.ProblemSet`s with {meth}`.create_problem_sets` that you can further configure if you wish.


In [6]:
from qrules.settings import InteractionType

stm = qrules.StateTransitionManager(
    initial_state=["J/psi(1S)"],
    final_state=["K0", "Sigma+", "p~"],
    formalism="canonical-helicity",
)
stm.set_allowed_interaction_types([InteractionType.STRONG, InteractionType.EM])
problem_sets = stm.create_problem_sets()

Note that the output of {meth}`.create_problem_sets` is a {obj}`dict` with {obj}`float` values as keys (representing the interaction strength) and {obj}`list`s of {obj}`.ProblemSet`s as values.


In [7]:
sorted(problem_sets, reverse=True)

[3600.0, 60.0, 1.0]

In [8]:
problem_set = problem_sets[60.0][0]
show_mermaid_markdown(problem_set, render_node=True)

```mermaid
flowchart LR
    n_0["0: K0[0]"]
    n_1["1: Sigma+[-1/2]"]
    n_2["2: p~[-1/2]"]
    A["J/psi(1S)[-1]"]
    N0["RULES<br/>clebsch_gordan_helicity_to_canonical - NA<br/>BaryonNumberConservation - 90<br/>ls_spin_validity - 89<br/>spin_magnitude_conservation - 8<br/>CharmConservation - 70<br/>helicity_conservation - 7<br/>StrangenessConservation - 69<br/>BottomnessConservation - 68<br/>isospin_conservation - 60<br/>parity_conservation - 6<br/>c_parity_conservation - 5<br/>ElectronLNConservation - 45<br/>MuonLNConservation - 44<br/>TauLNConservation - 43<br/>parity_conservation_helicity - 4<br/>g_parity_conservation - 3<br/>identical_particle_symmetrization - 2<br/>ChargeConservation - 100<br/>MassConservation - 10<br/>DOMAINS<br/>l_magnitude ∊ ［0, 1］<br/>l_projection ∊ ［0］<br/>parity_prefactor ∊ ［-1, +1］<br/>s_magnitude ∊ ［0, 1/2, 1, 3/2, 2］<br/>s_projection ∊ ［-2, -3/2, -1, -1/2, 0, +1/2, +1, +3/2, +2］"]
    N1["RULES<br/>clebsch_gordan_helicity_to_canonical - NA<br/>BaryonNumberConservation - 90<br/>ls_spin_validity - 89<br/>spin_magnitude_conservation - 8<br/>CharmConservation - 70<br/>helicity_conservation - 7<br/>StrangenessConservation - 69<br/>BottomnessConservation - 68<br/>parity_conservation - 6<br/>c_parity_conservation - 5<br/>ElectronLNConservation - 45<br/>MuonLNConservation - 44<br/>TauLNConservation - 43<br/>parity_conservation_helicity - 4<br/>identical_particle_symmetrization - 2<br/>ChargeConservation - 100<br/>MassConservation - 10<br/>DOMAINS<br/>l_magnitude ∊ ［0, 1］<br/>l_projection ∊ ［0］<br/>parity_prefactor ∊ ［-1, +1］<br/>s_magnitude ∊ ［0, 1/2, 1, 3/2, 2］<br/>s_projection ∊ ［-2, -3/2, -1, -1/2, 0, +1/2, +1, +3/2, +2］"]
    A --> N0
    N0 -->|RULES<br/>spin_validity - 62<br/>isospin_validity - 61<br/>gellmann_nishijima - 50<br/>DOMAINS<br/>baryon_number ∊ ［-1, 0, +1］<br/>bottomness ∊ ［-1, 0, +1］<br/>c_parity ∊ ［-1, +1, None］<br/>charge ∊ ［-2, -1, 0, +1, +2］<br/>charmness ∊ ［-1, 0, +1］<br/>electron_lepton_number ∊ ［-1, 0, +1］<br/>g_parity ∊ ［-1, +1, None］<br/>isospin_magnitude ∊ ［0, 1/2, 1, 3/2］<br/>isospin_projection ∊ ［-3/2, -1, -1/2, 0, +1/2, +1, +3/2］<br/>muon_lepton_number ∊ ［-1, 0, +1］<br/>parity ∊ ［-1, +1］<br/>spin_magnitude ∊ ［0, 1/2, 1, 3/2, 2, 5/2, 3, 7/2, 4］<br/>spin_projection ∊ ［-4, -7/2, -3, -5/2, -2, -3/2, -1, -1/2, 0, +1/2, +1, +3/2, +2, +5/2, +3, +7/2, +4］<br/>strangeness ∊ ［-3, -2, -1, 0, +1, +2, +3］<br/>tau_lepton_number ∊ ［-1, 0, +1］| N1
    N0 --> n_0
    N1 --> n_1
    N1 --> n_2

```

## Quantum number solutions


As noted in {ref}`usage/reaction:3. Find solutions`, a {obj}`.ProblemSet` can be fed to {meth}`.StateTransitionManager.find_solutions` directly to get a {obj}`.ReactionInfo` object. {obj}`.ReactionInfo` is a final result that consists of {obj}`.Particle`s, but in the intermediate steps, QRules works with sets of quantum numbers. One can inspect these intermediate generated quantum numbers by using {meth}`.find_quantum_number_transitions` and inspecting is output. Note that the resulting object is again a {obj}`dict` with strengths as keys and a list of solution as values.


In [9]:
qn_solutions = stm.find_quantum_number_transitions(problem_sets)
{strength: len(values) for strength, values in qn_solutions.items()}

Propagating quantum numbers:   0%|          | 0/144 [00:00<?, ?it/s]

{3600.0: 36, 60.0: 72, 1.0: 36}

The list of solutions consist of a {obj}`tuple` of a {obj}`.QNProblemSet` (compare {ref}`problem-sets`) and a {obj}`.QNResult`:


In [10]:
strong_qn_solutions = qn_solutions[3600.0]
qn_problem_set, qn_result = strong_qn_solutions[0]

In [11]:
show_mermaid_markdown(qn_problem_set, render_node=True)

```mermaid
flowchart LR
    n_0["0:<br/>pid = 311<br/>spin_magnitude = 0<br/>mass = 0.49761099999999997<br/>strangeness = +1<br/>parity = -1<br/>spin_projection = 0<br/>isospin_magnitude = 1/2<br/>isospin_projection = -1/2"]
    n_1["1:<br/>pid = 3222<br/>spin_magnitude = 1/2<br/>mass = 1.1893699999999998<br/>width = 8.208999999999999e-15<br/>charge = +1<br/>strangeness = -1<br/>baryon_number = +1<br/>parity = +1<br/>spin_projection = +1/2<br/>isospin_magnitude = 1<br/>isospin_projection = +1"]
    n_2["2:<br/>pid = -2212<br/>spin_magnitude = 1/2<br/>mass = 0.9382720894300001<br/>charge = -1<br/>baryon_number = -1<br/>parity = -1<br/>spin_projection = -1/2<br/>isospin_magnitude = 1/2<br/>isospin_projection = -1/2"]
    A["pid = 443<br/>spin_magnitude = 1<br/>mass = 3.0969<br/>width = 9.26e-05<br/>parity = -1<br/>c_parity = -1<br/>g_parity = -1<br/>spin_projection = -1<br/>isospin_magnitude = 0<br/>isospin_projection = 0"]
    N0["RULES<br/>clebsch_gordan_helicity_to_canonical - NA<br/>BaryonNumberConservation - 90<br/>ls_spin_validity - 89<br/>spin_magnitude_conservation - 8<br/>CharmConservation - 70<br/>helicity_conservation - 7<br/>StrangenessConservation - 69<br/>BottomnessConservation - 68<br/>isospin_conservation - 60<br/>parity_conservation - 6<br/>c_parity_conservation - 5<br/>ElectronLNConservation - 45<br/>MuonLNConservation - 44<br/>TauLNConservation - 43<br/>parity_conservation_helicity - 4<br/>g_parity_conservation - 3<br/>identical_particle_symmetrization - 2<br/>ChargeConservation - 100<br/>MassConservation - 10<br/>DOMAINS<br/>l_magnitude ∊ ［0, 1］<br/>l_projection ∊ ［0］<br/>parity_prefactor ∊ ［-1, +1］<br/>s_magnitude ∊ ［0, 1/2, 1, 3/2, 2］<br/>s_projection ∊ ［-2, -3/2, -1, -1/2, 0, +1/2, +1, +3/2, +2］"]
    N1["RULES<br/>clebsch_gordan_helicity_to_canonical - NA<br/>BaryonNumberConservation - 90<br/>ls_spin_validity - 89<br/>spin_magnitude_conservation - 8<br/>CharmConservation - 70<br/>helicity_conservation - 7<br/>StrangenessConservation - 69<br/>BottomnessConservation - 68<br/>isospin_conservation - 60<br/>parity_conservation - 6<br/>c_parity_conservation - 5<br/>ElectronLNConservation - 45<br/>MuonLNConservation - 44<br/>TauLNConservation - 43<br/>parity_conservation_helicity - 4<br/>g_parity_conservation - 3<br/>identical_particle_symmetrization - 2<br/>ChargeConservation - 100<br/>MassConservation - 10<br/>DOMAINS<br/>l_magnitude ∊ ［0, 1］<br/>l_projection ∊ ［0］<br/>parity_prefactor ∊ ［-1, +1］<br/>s_magnitude ∊ ［0, 1/2, 1, 3/2, 2］<br/>s_projection ∊ ［-2, -3/2, -1, -1/2, 0, +1/2, +1, +3/2, +2］"]
    A --> N0
    N0 -->|RULES<br/>spin_validity - 62<br/>isospin_validity - 61<br/>gellmann_nishijima - 50<br/>DOMAINS<br/>baryon_number ∊ ［-1, 0, +1］<br/>bottomness ∊ ［-1, 0, +1］<br/>c_parity ∊ ［-1, +1, None］<br/>charge ∊ ［-2, -1, 0, +1, +2］<br/>charmness ∊ ［-1, 0, +1］<br/>electron_lepton_number ∊ ［-1, 0, +1］<br/>g_parity ∊ ［-1, +1, None］<br/>isospin_magnitude ∊ ［0, 1/2, 1, 3/2］<br/>isospin_projection ∊ ［-3/2, -1, -1/2, 0, +1/2, +1, +3/2］<br/>muon_lepton_number ∊ ［-1, 0, +1］<br/>parity ∊ ［-1, +1］<br/>spin_magnitude ∊ ［0, 1/2, 1, 3/2, 2, 5/2, 3, 7/2, 4］<br/>spin_projection ∊ ［-4, -7/2, -3, -5/2, -2, -3/2, -1, -1/2, 0, +1/2, +1, +3/2, +2, +5/2, +3, +7/2, +4］<br/>strangeness ∊ ［-3, -2, -1, 0, +1, +2, +3］<br/>tau_lepton_number ∊ ［-1, 0, +1］| N1
    N0 --> n_0
    N1 --> n_1
    N1 --> n_2

```

In [12]:
show_mermaid_markdown(qn_result, render_node=True)

```mermaid
flowchart LR
    T0_0["0:<br/>pid = 311<br/>spin_magnitude = 0<br/>mass = 0.49761099999999997<br/>strangeness = +1<br/>parity = -1<br/>spin_projection = 0<br/>isospin_magnitude = 1/2<br/>isospin_projection = -1/2"]
    T0_1["1:<br/>pid = 3222<br/>spin_magnitude = 1/2<br/>mass = 1.1893699999999998<br/>width = 8.208999999999999e-15<br/>charge = +1<br/>strangeness = -1<br/>baryon_number = +1<br/>parity = +1<br/>spin_projection = +1/2<br/>isospin_magnitude = 1<br/>isospin_projection = +1"]
    T0_2["2:<br/>pid = -2212<br/>spin_magnitude = 1/2<br/>mass = 0.9382720894300001<br/>charge = -1<br/>baryon_number = -1<br/>parity = -1<br/>spin_projection = -1/2<br/>isospin_magnitude = 1/2<br/>isospin_projection = -1/2"]
    T0_A["pid = 443<br/>spin_magnitude = 1<br/>mass = 3.0969<br/>width = 9.26e-05<br/>parity = -1<br/>c_parity = -1<br/>g_parity = -1<br/>spin_projection = -1<br/>isospin_magnitude = 0<br/>isospin_projection = 0"]
    T0_N0["l_magnitude = 1<br/>s_magnitude = 1<br/>l_projection = 0<br/>s_projection = +1<br/>parity_prefactor = -1"]
    T0_N1["l_magnitude = 0<br/>s_magnitude = 1<br/>l_projection = 0<br/>s_projection = +1<br/>parity_prefactor = +1"]
    T0_A --> T0_N0
    T0_N0 -->|spin_magnitude = 1<br/>spin_projection = -1<br/>parity = -1<br/>isospin_magnitude = 1/2<br/>isospin_projection = +1/2<br/>strangeness = -1<br/>pid = -30313<br/>mass = 1.718<br/>width = 0.32| T0_N1
    T0_N0 --> T0_0
    T0_N1 --> T0_1
    T0_N1 --> T0_2
    T1_0["0:<br/>pid = 311<br/>spin_magnitude = 0<br/>mass = 0.49761099999999997<br/>strangeness = +1<br/>parity = -1<br/>spin_projection = 0<br/>isospin_magnitude = 1/2<br/>isospin_projection = -1/2"]
    T1_1["1:<br/>pid = 3222<br/>spin_magnitude = 1/2<br/>mass = 1.1893699999999998<br/>width = 8.208999999999999e-15<br/>charge = +1<br/>strangeness = -1<br/>baryon_number = +1<br/>parity = +1<br/>spin_projection = +1/2<br/>isospin_magnitude = 1<br/>isospin_projection = +1"]
    T1_2["2:<br/>pid = -2212<br/>spin_magnitude = 1/2<br/>mass = 0.9382720894300001<br/>charge = -1<br/>baryon_number = -1<br/>parity = -1<br/>spin_projection = -1/2<br/>isospin_magnitude = 1/2<br/>isospin_projection = -1/2"]
    T1_A["pid = 443<br/>spin_magnitude = 1<br/>mass = 3.0969<br/>width = 9.26e-05<br/>parity = -1<br/>c_parity = -1<br/>g_parity = -1<br/>spin_projection = -1<br/>isospin_magnitude = 0<br/>isospin_projection = 0"]
    T1_N0["l_magnitude = 1<br/>s_magnitude = 1<br/>l_projection = 0<br/>s_projection = -1<br/>parity_prefactor = -1"]
    T1_N1["l_magnitude = 0<br/>s_magnitude = 1<br/>l_projection = 0<br/>s_projection = +1<br/>parity_prefactor = +1"]
    T1_A --> T1_N0
    T1_N0 -->|spin_magnitude = 1<br/>spin_projection = +1<br/>parity = -1<br/>isospin_magnitude = 1/2<br/>isospin_projection = +1/2<br/>strangeness = -1<br/>pid = -30313<br/>mass = 1.718<br/>width = 0.32| T1_N1
    T1_N0 --> T1_0
    T1_N1 --> T1_1
    T1_N1 --> T1_2

```

### Filtering quantum number problem sets


Sometimes, only a certain subset of quantum numbers and conservation rules are relevant, or the number of solutions the {class}`.StateTransitionManager` gives by default is too large for the follow-up analysis.
The {func}`.filter_quantum_number_problem_set` function can be used to produce a {class}`.QNProblemSet` where only the desired quantum numbers and conservation rules are considered when fed back to the solver.


In [13]:
desired_edge_properties = {EdgeQuantumNumbers.spin_magnitude, EdgeQuantumNumbers.parity}
desired_node_properties = {
    NodeQuantumNumbers.l_magnitude,
    NodeQuantumNumbers.s_magnitude,
}  # has to be reused in the CSPSolver-constructor
filtered_qn_problem_set = filter_quantum_number_problem_set(
    qn_problem_set,
    edge_rules={spin_validity},
    node_rules={spin_magnitude_conservation, parity_conservation},
    edge_properties=desired_edge_properties,
    node_properties=desired_node_properties,
)

In [14]:
show_mermaid_markdown(filtered_qn_problem_set, render_node=True)

```mermaid
flowchart LR
    n_0["0:<br/>spin_magnitude = 0<br/>parity = -1"]
    n_1["1:<br/>spin_magnitude = 1/2<br/>parity = +1"]
    n_2["2:<br/>spin_magnitude = 1/2<br/>parity = -1"]
    A["spin_magnitude = 1<br/>parity = -1"]
    N0["RULES<br/>spin_magnitude_conservation - 8<br/>parity_conservation - 6<br/>DOMAINS<br/>l_magnitude ∊ ［0, 1］<br/>s_magnitude ∊ ［0, 1/2, 1, 3/2, 2］"]
    N1["RULES<br/>spin_magnitude_conservation - 8<br/>parity_conservation - 6<br/>DOMAINS<br/>l_magnitude ∊ ［0, 1］<br/>s_magnitude ∊ ［0, 1/2, 1, 3/2, 2］"]
    A --> N0
    N0 -->|RULES<br/>spin_validity - 62<br/>DOMAINS<br/>parity ∊ ［-1, +1］<br/>spin_magnitude ∊ ［0, 1/2, 1, 3/2, 2, 5/2, 3, 7/2, 4］| N1
    N0 --> n_0
    N1 --> n_1
    N1 --> n_2

```

:::{warning}
The next cell will use some (currently) internal functionality. As stated at the top, a workflow similar to this will be used in future versions of {mod}`qrules`, see e.g. [ComPWA/qrules#305](https://github.com/ComPWA/qrules/issues/305). Manual setup of the {obj}`.CSPSolver` like in here will then also not be necessary.
:::


In [15]:
solver = CSPSolver([
    dict_set_intersection(
        qrules.system_control.create_edge_properties(part),
        desired_edge_properties,
    )
    for part in qrules.particle.load_pdg()
])

filtered_qn_solutions = solver.find_solutions(filtered_qn_problem_set)
filtered_qn_result = filtered_qn_solutions.solutions[6]

In [16]:
show_mermaid_markdown(filtered_qn_result, render_node=True)

```mermaid
flowchart LR
    n_0["0:<br/>spin_magnitude = 0<br/>parity = -1"]
    n_1["1:<br/>spin_magnitude = 1/2<br/>parity = +1"]
    n_2["2:<br/>spin_magnitude = 1/2<br/>parity = -1"]
    A["spin_magnitude = 1<br/>parity = -1"]
    N0["l_magnitude = 1<br/>s_magnitude = 1"]
    N1["l_magnitude = 0<br/>s_magnitude = 1"]
    A --> N0
    N0 -->|parity = -1<br/>spin_magnitude = 1| N1
    N0 --> n_0
    N1 --> n_1
    N1 --> n_2

```

## {obj}`.StateTransition`s


After finding the {ref}`usage/visualize:Quantum number solutions`, QRules finds {obj}`.Particle` definitions that match these quantum numbers. All these steps are hidden in the convenience functions {meth}`.StateTransitionManager.find_solutions` and {func}`.generate_transitions`. In the following, we'll visualize the allowed transitions for the decay $\psi' \to \gamma\eta\eta$ as an example.


In [17]:
import qrules

reaction = qrules.generate_transitions(
    initial_state="psi(2S)",
    final_state=["gamma", "eta", "eta"],
    allowed_interaction_types="EM",
)

Propagating quantum numbers:   0%|          | 0/12 [00:00<?, ?it/s]

As noted in {ref}`usage/reaction:3. Find solutions`, the {attr}`~.ReactionInfo.transitions` contain all spin projection combinations (which is necessary for the {mod}`ampform` package). It is possible to convert all these solutions to Mermaid markdown language with {func}`~.asmermaid`. To avoid visualizing all solutions with {func}`~.show_mermaid_markdown`, we just take a subset of the {attr}`~.ReactionInfo.transitions`:


In [18]:
show_mermaid_markdown(
    reaction.transitions[::50][:3], render_node=False
)  # just some selection

```mermaid
flowchart LR
    T0_0["0: gamma[-1]"]
    T0_1["1: eta[0]"]
    T0_2["2: eta[0]"]
    T0_A["psi(2S)[-1]"]
    T0_N0
    T0_N1
    T0_A --> T0_N0
    T0_N0 -->|"f(2)(2340)[-2]"| T0_N1
    T0_N0 --> T0_0
    T0_N1 --> T0_1
    T0_N1 --> T0_2
    T1_0["0: gamma[-1]"]
    T1_1["1: eta[0]"]
    T1_2["2: eta[0]"]
    T1_A["psi(2S)[-1]"]
    T1_N0
    T1_N1
    T1_A --> T1_N0
    T1_N0 -->|"f(2)'(1525)[0]"| T1_N1
    T1_N0 --> T1_0
    T1_N1 --> T1_1
    T1_N1 --> T1_2
    T2_0["0: gamma[-1]"]
    T2_1["1: eta[0]"]
    T2_2["2: eta[0]"]
    T2_A["psi(2S)[-1]"]
    T2_N0
    T2_N1
    T2_A --> T2_N0
    T2_N0 -->|"a(0)(980)0[0]"| T2_N1
    T2_N0 --> T2_0
    T2_N1 --> T2_1
    T2_N1 --> T2_2

```

You can also serialize the Markdown string to file with {func}`.io.write`. The file extension for a Markdown file is `.md`:


In [19]:
qrules.io.write(reaction, "decay_topologies_with_spin.md")

You can also use the file extension `.mmd` for Mermaid files:


In [20]:
qrules.io.write(reaction, "decay_topologies_with_spin.mmd")

### Collapse graphs


Since this list of all possible spin projections {attr}`~.ReactionInfo.transitions` is rather long, it is often useful to use `strip_spin=True` or `collapse_graphs=True` to bundle comparable graphs. First, {code}`strip_spin=True` allows one collapse (ignore) the spin projections (we again show a selection only):


In [21]:
show_mermaid_markdown(reaction.transitions[:3], strip_spin=True)

```mermaid
flowchart LR
    T0_0["0: gamma"]
    T0_1["1: eta"]
    T0_2["2: eta"]
    T0_A["psi(2S)"]
    T0_N0
    T0_N1
    T0_A --> T0_N0
    T0_N0 -->|"a(2)(1320)0"| T0_N1
    T0_N0 --> T0_0
    T0_N1 --> T0_1
    T0_N1 --> T0_2
    T1_0["0: gamma"]
    T1_1["1: eta"]
    T1_2["2: eta"]
    T1_A["psi(2S)"]
    T1_N0
    T1_N1
    T1_A --> T1_N0
    T1_N0 -->|"a(0)(980)0"| T1_N1
    T1_N0 --> T1_0
    T1_N1 --> T1_1
    T1_N1 --> T1_2

```

or, with stripped node properties:


In [22]:
show_mermaid_markdown(reaction.transitions[:3], strip_spin=True, render_node=True)

```mermaid
flowchart LR
    T0_0["0: gamma"]
    T0_1["1: eta"]
    T0_2["2: eta"]
    T0_A["psi(2S)"]
    T0_N0["L=0<br/>S=1<br/>P=+1"]
    T0_N1["L=2<br/>S=0<br/>P=+1"]
    T0_A --> T0_N0
    T0_N0 -->|"a(2)(1320)0"| T0_N1
    T0_N0 --> T0_0
    T0_N1 --> T0_1
    T0_N1 --> T0_2
    T1_0["0: gamma"]
    T1_1["1: eta"]
    T1_2["2: eta"]
    T1_A["psi(2S)"]
    T1_N0["L=2<br/>S=1<br/>P=+1"]
    T1_N1["L=0<br/>S=0<br/>P=+1"]
    T1_A --> T1_N0
    T1_N0 -->|"a(0)(980)0"| T1_N1
    T1_N0 --> T1_0
    T1_N1 --> T1_1
    T1_N1 --> T1_2
    T2_0["0: gamma"]
    T2_1["1: eta"]
    T2_2["2: eta"]
    T2_A["psi(2S)"]
    T2_N0["L=0<br/>S=1<br/>P=+1"]
    T2_N1["L=0<br/>S=0<br/>P=+1"]
    T2_A --> T2_N0
    T2_N0 -->|"a(0)(980)0"| T2_N1
    T2_N0 --> T2_0
    T2_N1 --> T2_1
    T2_N1 --> T2_2

```

```{note}
By default, {func}`.asmermaid` renders edge IDs, because they represent the (final) state IDs as well. In the example above, we switched this off.
```


If that list is still too much, there is {code}`collapse_graphs=True`, which bundles all graphs with the same final state groupings:


In [23]:
show_mermaid_markdown(reaction, collapse_graphs=True, render_node=False)

```mermaid
flowchart LR
    T0_0["0: gamma"]
    T0_1["1: eta"]
    T0_2["2: eta"]
    T0_A["psi(2S)"]
    T0_N0
    T0_N1
    T0_A --> T0_N0
    T0_N0 -->|"b(1)(1235)0<br/>h(1)(1170)<br/>h(1)(1415)<br/>J/psi(1S)<br/>omega(782)<br/>omega(1420)<br/>omega(1650)<br/>phi(1020)<br/>phi(1680)<br/>rho(770)0<br/>rho(1450)0<br/>rho(1700)0"| T0_N1
    T0_N0 --> T0_1
    T0_N1 --> T0_0
    T0_N1 --> T0_2
    T1_0["0: gamma"]
    T1_1["1: eta"]
    T1_2["2: eta"]
    T1_A["psi(2S)"]
    T1_N0
    T1_N1
    T1_A --> T1_N0
    T1_N0 -->|"a(0)(980)0<br/>a(2)(1320)0<br/>a(0)(1450)0<br/>a(2)(1700)0<br/>chi(c0)(1P)<br/>chi(c2)(1P)<br/>f(0)(500)<br/>f(0)(980)<br/>f(2)(1270)<br/>f(0)(1370)<br/>f(2)'(1525)<br/>f(0)(1500)<br/>f(2)(1565)<br/>f(0)(1710)<br/>f(2)(1950)<br/>f(0)(2020)<br/>f(2)(2010)<br/>f(2)(2150)<br/>f(2)(2300)<br/>f(2)(2340)"| T1_N1
    T1_N0 --> T1_0
    T1_N1 --> T1_1
    T1_N1 --> T1_2

```

### Other state renderings


The {meth}`~.FrozenTransition.convert` method makes it possible to convert the types of its {attr}`~.FrozenTransition.states`. This for instance allows us to only render the spin states on in a {class}`.Transition`:

::::{margin}

:::{tip}

We use the fact that a {obj}`.StateTransition` is frozen (and therefore hashable) to remove any duplicate transitions.

:::

::::


In [24]:
spin_transitions = sorted({
    t.convert(lambda s: Spin(s.particle.spin, s.spin_projection))
    for t in reaction.transitions
})
some_selection = spin_transitions[::67][:3]
show_mermaid_markdown(some_selection, render_node=True)

```mermaid
flowchart LR
    T0_0["0: |1,+1⟩"]
    T0_1["1: |0,0⟩"]
    T0_2["2: |0,0⟩"]
    T0_A["|1,+1⟩"]
    T0_N0["L=|2,0⟩<br/>S=|1,+1⟩<br/>P=+1"]
    T0_N1["L=|2,0⟩<br/>S=|1,+1⟩<br/>P=+1"]
    T0_A --> T0_N0
    T0_N0 --"|1,-1⟩"--> T0_N1
    T0_N0 --> T0_1
    T0_N1 --> T0_0
    T0_N1 --> T0_2
    T1_0["0: |1,-1⟩"]
    T1_1["1: |0,0⟩"]
    T1_2["2: |0,0⟩"]
    T1_A["|1,-1⟩"]
    T1_N0["L=|2,0⟩<br/>S=|1,0⟩<br/>P=+1"]
    T1_N1["L=|0,0⟩<br/>S=|1,-1⟩<br/>P=+1"]
    T1_A --> T1_N0
    T1_N0 --"|1,0⟩"--> T1_N1
    T1_N0 --> T1_1
    T1_N1 --> T1_0
    T1_N1 --> T1_2
    T2_0["0: |1,-1⟩"]
    T2_1["1: |0,0⟩"]
    T2_2["2: |0,0⟩"]
    T2_A["|1,-1⟩"]
    T2_N0["L=|0,0⟩<br/>S=|1,-1⟩<br/>P=+1"]
    T2_N1["L=|0,0⟩<br/>S=|0,0⟩<br/>P=+1"]
    T2_A --> T2_N0
    T2_N0 --"|0,0⟩"--> T2_N1
    T2_N0 --> T2_0
    T2_N1 --> T2_1
    T2_N1 --> T2_2

```

Or any other properties of a {class}`.State`, such as masses or $J^{PC}(I^G)$ numbers:


In [25]:
def render_mass(state: State, digits: int = 3) -> str:
    mass = round(state.particle.mass, digits)
    width = round(state.particle.width, digits)
    if width == 0:
        return str(mass)
    return f"{mass}±{width}"


mass_transitions = sorted({
    t.convert(
        state_converter=render_mass,
        interaction_converter=lambda _: None,
    )
    for t in reaction.transitions
})
show_mermaid_markdown(mass_transitions[::10])

```mermaid
flowchart LR
    T0_0["0: 0.0"]
    T0_1["1: 0.548"]
    T0_2["2: 0.548"]
    T0_A["3.686"]
    T0_N0
    T0_N1
    T0_A --> T0_N0
    T0_N0 -->|1.72±0.25| T0_N1
    T0_N0 --> T0_1
    T0_N1 --> T0_0
    T0_N1 --> T0_2
    T1_0["0: 0.0"]
    T1_1["1: 0.548"]
    T1_2["2: 0.548"]
    T1_A["3.686"]
    T1_N0
    T1_N1
    T1_A --> T1_N0
    T1_N0 -->|0.775±0.147| T1_N1
    T1_N0 --> T1_1
    T1_N1 --> T1_0
    T1_N1 --> T1_2
    T2_0["0: 0.0"]
    T2_1["1: 0.548"]
    T2_2["2: 0.548"]
    T2_A["3.686"]
    T2_N0
    T2_N1
    T2_A --> T2_N0
    T2_N0 -->|1.706±0.38| T2_N1
    T2_N0 --> T2_0
    T2_N1 --> T2_1
    T2_N1 --> T2_2
    T3_0["0: 0.0"]
    T3_1["1: 0.548"]
    T3_2["2: 0.548"]
    T3_A["3.686"]
    T3_N0
    T3_N1
    T3_A --> T3_N0
    T3_N0 -->|0.6±0.45| T3_N1
    T3_N0 --> T3_0
    T3_N1 --> T3_1
    T3_N1 --> T3_2

```

In [26]:
from fractions import Fraction


def render_jpc_ig(state: State) -> str:
    particle = state.particle
    text = render_fraction(particle.spin)
    if particle.parity is not None:
        text += render_sign(particle.parity)
    if particle.c_parity is not None:
        text += render_sign(particle.c_parity)
    if particle.isospin is not None and particle.g_parity is not None:
        text += "("
        text += f"{render_fraction(particle.isospin.magnitude)}"
        text += f"{render_sign(particle.g_parity)}"
        text += ")"
    return text


def render_fraction(value: float) -> str:
    fraction = Fraction(value)
    if fraction.denominator == 1:
        return str(fraction.numerator)
    return f"{fraction.numerator}/{fraction.denominator}"


def render_sign(parity: int) -> str:
    if parity == -1:
        return "⁻"
    if parity == +1:
        return "⁺"
    raise NotImplementedError


jpc_ig_transitions = sorted({
    t.convert(
        state_converter=render_jpc_ig,
        interaction_converter=lambda _: None,
    )
    for t in reaction.transitions
})
show_mermaid_markdown(jpc_ig_transitions, collapse_graphs=True)

```mermaid
flowchart LR
    T0_0["0: 1⁻⁻"]
    T0_1["1: 0⁻⁺(0⁺)"]
    T0_2["2: 0⁻⁺(0⁺)"]
    T0_A["1⁻⁻(0⁻)"]
    T0_N0
    T0_N1
    T0_A --> T0_N0
    T0_N0 -->|"1⁺⁻(0⁻)<br/>1⁺⁻(1⁺)<br/>1⁻⁻(0⁻)<br/>1⁻⁻(1⁺)"| T0_N1
    T0_N0 --> T0_1
    T0_N1 --> T0_0
    T0_N1 --> T0_2
    T1_0["0: 1⁻⁻"]
    T1_1["1: 0⁻⁺(0⁺)"]
    T1_2["2: 0⁻⁺(0⁺)"]
    T1_A["1⁻⁻(0⁻)"]
    T1_N0
    T1_N1
    T1_A --> T1_N0
    T1_N0 -->|"0⁺⁺(0⁺)<br/>0⁺⁺(1⁻)<br/>2⁺⁺(0⁺)<br/>2⁺⁺(1⁻)"| T1_N1
    T1_N0 --> T1_0
    T1_N1 --> T1_1
    T1_N1 --> T1_2

```

:::{tip}
Note that collapsing the graphs also works for custom edge properties.
:::


## Styling


<!-- cspell:ignore darkgreen fontcolor fontsize penwidth -->

The {func}`.asmermaid` and {func}`.show_mermaid_markdown` functions also accept styling arguments. Use `figure_style` for the whole diagram, `edge_style` for edges, and `node_style` for nodes:


In [27]:
show_mermaid_markdown(
    reaction.transitions[0],
    render_node=True,
    figure_style={"bgcolor": "white"},
    edge_style={
        "color": "red",
        "fontcolor": "blue",
        "fontsize": 25,
    },
    node_style={
        "color": "gray",
        "fill": "lightgray",
        "stroke": "black",
    },
)

```mermaid
flowchart LR
classDef default fill:white
    n_0["0: gamma[-1]"]
    n_1["1: eta[0]"]
    n_2["2: eta[0]"]
    A["psi(2S)[-1]"]
    N0["L=|0,0⟩<br/>S=|1,-1⟩<br/>P=+1"]
    N1["L=|0,0⟩<br/>S=|0,0⟩<br/>P=+1"]
    A --> N0
    N0 -->|"a(0)(980)0[0]"| N1
    N0 --> n_0
    N1 --> n_1
    N1 --> n_2
    style n_0 color:gray,fill:lightgray,stroke:black
    style n_1 color:gray,fill:lightgray,stroke:black
    style n_2 color:gray,fill:lightgray,stroke:black
    style A color:gray,fill:lightgray,stroke:black
    style N0 color:gray,fill:lightgray,stroke:black
    style N1 color:gray,fill:lightgray,stroke:black
    linkStyle 0 stroke:red,color:blue,font-size:25
    linkStyle 1 stroke:red,color:blue,font-size:25
    linkStyle 2 stroke:red,color:blue,font-size:25
    linkStyle 3 stroke:red,color:blue,font-size:25
    linkStyle 4 stroke:red,color:blue,font-size:25

```